In [2]:
import math
import warnings
from fractions import Fraction
 
import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix
from time import time
from tqdm import tqdm, trange
 
_NEG = np.iinfo(np.int64).min // 4  

Symmetry Resolution

1. x_{ij} = 0 forall j > i : Clf i cannot be put into a bucket numbered above itself

2. x_{ij} <= sum_{k < i} x_{k, j-1} forall j >= 1 : bucket j can only be used once bucket j-1 is occupied by an earlier classifier

BR Tie Breaking
Calculate net utility as p_i * (h_i(a) - (1+eps)*c(x, a))



*** We can decrease the number of Variables ***
WLOG assume i=1 is in j=1, i=2 in j=1,2 and i=3 in j=1,2,3

*** Remove every variable with j > i ***
The partition achieved is unique (no further refinement possible)
Going from n^n to n!

Write down why this works?
Only unique one solution that exists. Cannot prune more. 


*** Implement Quadratic Solver ***

Compare hard instances of greedy with MILP
Choose specific time limit for MILP and compare with greedy
With maybe n=100

1. Run greedy on random instances and hard greedy instances and compare with Branch and Bound
    a. Need extensive experiments (comparisons)
    b. Give the same time limit to both algs
    c. Grid search for greedy and MILP (original and reduced variables)

*** Use cvxpy instead of scipy ***
cvxpy has quadratic solvers


Play around if this program can be written fro Uniform dist instead of a discrete dist

In [3]:
[2,...], [1, ...]

([2, Ellipsis], [1, Ellipsis])

In [4]:
def rationalize(values, max_denominator, name, snap_tol=1e-9):
    out, worst = [], 0.0
    for v in np.atleast_1d(np.asarray(values, dtype=float)).ravel():
        if not math.isfinite(v):
            raise ValueError(f"{name} contains non-finite value {v}")
        f = Fraction(float(v)).limit_denominator(max_denominator)
        worst = max(worst, abs(float(f) - float(v)))
        out.append(f)
    if worst > snap_tol:
        warnings.warn(
            f"{name}: rational snapping moved a value by {worst:.2e} "
            f"(> {snap_tol:.0e}). Raise max_denominator, or pass values that "
            f"are exactly representable as simple fractions.", stacklevel=3)
    return out

In [5]:
def lcm_denominators(fracs):
    q = 1
    for f in fracs:
        q = q * f.denominator // math.gcd(q, f.denominator)
    return q

In [6]:
def build_action_sets(t, X):
    n, m = len(t), len(X)
    acts = np.empty((m, n + 1), dtype=object)
    nact = np.zeros(m, dtype=np.int64)
    for xi, x in enumerate(X):
        A = sorted({x} | {ti for ti in t if ti > x})
        nact[xi] = len(A)
        for ai, a in enumerate(A):
            acts[xi, ai] = a
    return acts, nact

In [7]:
def build_integer_tables(t, pi, tau, c, X, acts, nact):
    n, m = len(t), len(X)
    Q = lcm_denominators(pi) * c.denominator * lcm_denominators(list(t) + list(X))
    limit = np.iinfo(np.int64).max // max(n, 1)
 
    Uint = np.zeros((n, m, n + 1), dtype=np.int64)
    Lmat = np.zeros((n, m, n + 1), dtype=np.int8)
    valid = np.zeros((m, n + 1), dtype=bool)
    for xi in range(m):
        label = 1 if X[xi] >= tau else 0
        for ai in range(int(nact[xi])):
            a = acts[xi, ai]
            valid[xi, ai] = True
            cost = c * max(a - X[xi], Fraction(0))
            for i in range(n):
                h = 1 if a >= t[i] else 0
                u = pi[i] * (h - cost) * Q
                if u.denominator != 1:
                    raise AssertionError("integer scaling failed")
                if abs(int(u)) > limit:
                    raise OverflowError(
                        "scaled utilities exceed int64 range; lower "
                        "max_denominator or use simpler rational inputs")
                Uint[i, xi, ai] = int(u)
                Lmat[i, xi, ai] = 1 if h != label else 0
    return Uint, Lmat, valid, Q

In [8]:
def block_response(Uint, valid, members):
    total = Uint[list(members)].sum(axis=0)
    return np.where(valid, total, _NEG).argmax(axis=1)

def block_loss(Uint, valid, Lmat, Dint, Pint, members):
    best = block_response(Uint, valid, members)
    rows = np.arange(Uint.shape[1])
    total = 0
    for i in members:
        hit = np.nonzero(Lmat[i, rows, best])[0]
        if hit.size:
            total += Pint[i] * sum(Dint[r] for r in hit)
    return total

def partition_loss(Uint, valid, Lmat, Dint, Pint, partition):
    return sum(block_loss(Uint, valid, Lmat, Dint, Pint, list(b)) for b in partition)

In [9]:
def solve_dp(n, Uint, valid, Lmat, Dint, Pint):
    full = (1 << n) - 1
 
    cost = [0] * (1 << n)
    for mask in range(1, 1 << n):
        members = [i for i in range(n) if mask >> i & 1]
        cost[mask] = block_loss(Uint, valid, Lmat, Dint, Pint, members)
 
    f = [None] * (1 << n)
    choice = [0] * (1 << n)
    f[0] = 0
    for mask in range(1, 1 << n):
        low = mask & -mask
        rest = mask ^ low
        sub = rest
        while True:
            S = sub | low
            prev = f[mask ^ S]
            if prev is not None:
                val = cost[S] + prev
                if f[mask] is None or val < f[mask]:
                    f[mask], choice[mask] = val, S
            if sub == 0:
                break
            sub = (sub - 1) & rest
 
    part, mask = [], full
    while mask:
        S = choice[mask]
        part.append(sorted(i for i in range(n) if S >> i & 1))
        mask ^= S
    return f[full], sorted(part)

In [10]:
def iter_set_partitions(elements):
    if not elements:
        yield []
        return
    first, rest = elements[0], elements[1:]
    for smaller in iter_set_partitions(rest):
        for k in range(len(smaller)):
            yield smaller[:k] + [[first] + smaller[k]] + smaller[k + 1:]
        yield [[first]] + smaller
 
def solve_brute(n, Uint, valid, Lmat, Dint, Pint):
    best_p = best_v = None
    for p in iter_set_partitions(list(range(n))):
        v = partition_loss(Uint, valid, Lmat, Dint, Pint, p)
        if best_v is None or v < best_v:
            best_p, best_v = p, v
    return best_v, sorted(sorted(b) for b in best_p)

In [ ]:
def certified_epsilon(Q, c, X, acts, nact):
    Cmax = Fraction(0)
    for xi in range(len(X)):
        for ai in range(int(nact[xi])):
            Cmax = max(Cmax, c * max(acts[xi, ai] - X[xi], Fraction(0)))
    return Fraction(1, 2) if Cmax == 0 else Fraction(1, 2) / (Q * Cmax)

def perturbed_utilities(t, pi, c, X, acts, nact, eps):
    n, m = len(t), len(X)
    Uf = np.zeros((n, m, n + 1))
    for xi in range(m):
        for ai in range(int(nact[xi])):
            cost = c * max(acts[xi, ai] - X[xi], Fraction(0))
            for i in range(n):
                h = 1 if acts[xi, ai] >= t[i] else 0
                Uf[i, xi, ai] = float(pi[i] * (h - (1 + eps) * cost))
    return Uf


def solve_milp(n, m, Uf, nact, piL, Dfloat, time_limit=None, verbose=False, mip_rel_gap=0.1):
    off_x = 0
    off_y = off_x + n * n
    yoff, cur = [], off_y
    for xi in range(m):
        yoff.append(cur)
        cur += int(nact[xi]) * n
    off_v = cur
    off_w = off_v + m * n
    nvar = off_w + m * n
 
    def ix(i, j):
        return off_x + i * n + j
 
    def iy(xi, ai, j):
        return yoff[xi] + ai * n + j
 
    def iv(xi, j):
        return off_v + xi * n + j
 
    def iw(xi, j):
        return off_w + xi * n + j
 
    rows, cols, vals, lb, ub = [], [], [], [], []
    row_count = 0
 
    def add(entries, lo, hi):
        nonlocal row_count
        for cix, cf in entries:
            rows.append(row_count)
            cols.append(cix)
            vals.append(float(cf))
        lb.append(lo)
        ub.append(hi)
        row_count += 1
 
    for i in range(n):
        add([(ix(i, j), 1.0) for j in range(n)], 1.0, 1.0)
 
    for i in range(1, n):
        for j in range(1, i + 1):
            add([(ix(i, j), 1.0)] + [(ix(ip, j - 1), -1.0) for ip in range(i)],
                -np.inf, 0.0)
 
    for xi in range(m):
        for j in range(n):
            add([(iy(xi, ai, j), 1.0) for ai in range(int(nact[xi]))], 1.0, 1.0)
 
    for xi in range(m):
        for j in range(n):
            for ai in range(int(nact[xi])):
                ent = [(ix(i, j), -Uf[i, xi, ai]) for i in range(n)]
                add(ent + [(iv(xi, j), 1.0)], 0.0, np.inf)
                M = 0.0
                for ap in range(int(nact[xi])):
                    M = max(M, float(sum(max(Uf[i, xi, ap] - Uf[i, xi, ai], 0.0)
                                         for i in range(n))))
                ent2 = [(ix(i, j), Uf[i, xi, ai]) for i in range(n)]
                add(ent2 + [(iv(xi, j), -1.0), (iy(xi, ai, j), -M)], -M, np.inf)
 
    for xi in range(m):
        for j in range(n):
            for ai in range(int(nact[xi])):
                Mxa = float(piL[:, xi, ai].sum())
                ent = [(ix(i, j), -float(piL[i, xi, ai])) for i in range(n)
                       if piL[i, xi, ai] != 0.0]
                add(ent + [(iw(xi, j), 1.0), (iy(xi, ai, j), -Mxa)],
                    -Mxa, np.inf)
 
    A = coo_matrix((vals, (rows, cols)), shape=(row_count, nvar))
    obj = np.zeros(nvar)
    for xi in range(m):
        for j in range(n):
            obj[iw(xi, j)] = float(Dfloat[xi])
 
    vlb, vub = np.zeros(nvar), np.ones(nvar)
    for i in range(n):
        for j in range(i + 1, n):
            vub[ix(i, j)] = 0.0                  # symmetry: bucket j > i unused
    vlb[off_v:off_v + m * n] = -np.inf
    vub[off_v:off_v + m * n] = np.inf
    vub[off_w:off_w + m * n] = np.inf
 
    integrality = np.zeros(nvar)
    integrality[off_x:off_v] = 1
 
    # options = {"disp": bool(verbose), "mip_rel_gap": mip_rel_gap}
    options =  {"mip_rel_gap": 0.01, "disp": True} # Sets the relative gap to 1% "disp": True # Highly recommended: prints the gap log as it solves }
    if time_limit:
        options["time_limit"] = float(time_limit)
 
    res = milp(c=obj, constraints=LinearConstraint(A, lb, ub),
               integrality=integrality, bounds=Bounds(vlb, vub),
               options=options)
    if not res.success or res.x is None:
        raise RuntimeError(f"HiGHS did not solve: {res.message}")
 
    blocks = {}
    for i in range(n):
        for j in range(n):
            if res.x[ix(i, j)] > 0.5:
                blocks.setdefault(j, []).append(i)
    return sorted(sorted(b) for b in blocks.values()), nvar, row_count

In [12]:
def find_optimal_partition(thresholds, priors, true_threshold, c, X, weights=None, method="dp", epsilon="auto", max_denominator=10 ** 6, time_limit=None, verbose=False, check=False, mip_rel_gap=0.1):
    t_raw = np.asarray(thresholds, dtype=float).ravel()
    p_raw = np.asarray(priors, dtype=float).ravel()
    X_in = np.asarray(X, dtype=float).ravel()
    X_raw = np.unique(X_in)
    n, m = len(t_raw), len(X_raw)
    if len(p_raw) != n:
        raise ValueError(f"priors has length {len(p_raw)}, thresholds has {n}")
    if np.any(p_raw <= 0):
        raise ValueError("priors must be strictly positive")
    if n == 0 or m == 0:
        raise ValueError("thresholds and X must be non-empty")
 
    t = rationalize(t_raw, max_denominator, "thresholds")
    pi = rationalize(p_raw, max_denominator, "priors")
    sp = sum(pi)
    pi = [p / sp for p in pi]
    tau = rationalize([true_threshold], max_denominator, "true_threshold")[0]
    cc = rationalize([c], max_denominator, "c")[0]
    Xf = rationalize(X_raw, max_denominator, "X")
 
    if weights is None:
        D = [Fraction(1, m)] * m
    else:
        w_raw = np.asarray(weights, dtype=float).ravel()
        if len(w_raw) != len(X_in):
            raise ValueError("weights must have the same length as X")
        folded = {}
        for v, wv in zip(X_in, w_raw):
            folded[v] = folded.get(v, 0.0) + wv      # fold duplicate types
        w_aligned = np.array([folded[v] for v in X_raw])
        if np.any(w_aligned < 0):
            raise ValueError("weights must be non-negative")
        D = rationalize(w_aligned, max_denominator, "weights")
        sd = sum(D)
        D = [d / sd for d in D]
 
    acts, nact = build_action_sets(t, Xf)
    Uint, Lmat, valid, Q = build_integer_tables(t, pi, tau, cc, Xf, acts, nact)
 
    Ld, Lp = lcm_denominators(D), lcm_denominators(pi)
    Dint = [int(d * Ld) for d in D]
    Pint = [int(p * Lp) for p in pi]
    loss_scale = Ld * Lp
 
    eps_used = 0.0
    n_vars = n_constrs = 0
    if method == "dp":
        loss_int, part = solve_dp(n, Uint, valid, Lmat, Dint, Pint)
    elif method == "brute":
        loss_int, part = solve_brute(n, Uint, valid, Lmat, Dint, Pint)
    elif method == "milp":
        if isinstance(epsilon, str):
            eps = certified_epsilon(Q, cc, Xf, acts, nact)
            if float(eps) < 1e-7:
                warnings.warn(
                    f"certified epsilon is {float(eps):.2e}, small enough that "
                    f"the solver's integrality tolerance may swamp it; prefer "
                    f"method='dp' or simpler rational inputs.", stacklevel=2)
        else:
            eps = Fraction(float(epsilon)).limit_denominator(10 ** 12)
        eps_used = float(eps)
        Uf = perturbed_utilities(t, pi, cc, Xf, acts, nact, eps)
        piL = np.array([[[float(pi[i]) * int(Lmat[i, xi, ai])
                          for ai in range(n + 1)]
                         for xi in range(m)] for i in range(n)])
        Dfloat = np.array([float(d) for d in D])
        part, n_vars, n_constrs = solve_milp(n, m, Uf, nact, piL, Dfloat,
                                             time_limit, verbose, mip_rel_gap)
        # re-evaluate exactly: the MILP objective is float, the partition is not
        loss_int = partition_loss(Uint, valid, Lmat, Dint, Pint, part)
    else:
        raise ValueError("method must be 'dp', 'milp' or 'brute'")
 
    # if check and method != "dp":
    #     dp_loss, dp_part = solve_dp(n, Uint, valid, Lmat, Dint, Pint)
    #     if dp_loss != loss_int:
    #         raise AssertionError(
    #             f"{method} loss {loss_int} != dp {dp_loss} "
    #             f"({part} vs {dp_part})")
 
    loss_fraction = Fraction(int(loss_int), int(loss_scale))
 
    err = np.zeros(n)
    mass = []
    rows = np.arange(m)
    for block in part:
        mass.append(float(sum(pi[i] for i in block)))
        best = block_response(Uint, valid, list(block))
        for i in block:
            hit = np.nonzero(Lmat[i, rows, best])[0]
            err[i] = float(sum(D[xi] for xi in hit))
 
    return {"loss": float(loss_fraction),
            "partition": part,
            "per_classifier_error": err,
            "block_mass": np.array(mass),
            "method": method,
            "epsilon": eps_used,
            "n_vars": n_vars,
            "n_constrs": n_constrs,
            "priors_normalized": np.array([float(p) for p in pi]),
            "loss_fraction": loss_fraction}

In [13]:
rng = np.random.default_rng(11)
ok = bad = 0
# N = range(4, 10)
# M = [51, 101, 151, 201]
N = [8]
M = [10]
tt = 0.1
c = 0.1

res_stats = {"n": [], "m": [], "counter": [], "thresholds": [], "priors": [], "c": [], "tt": [], "alg": [], "time": [], "partition": [], "loss": [], "n_vars": [], "n_constrs": []}

for n in N:
    for m in M:
        X = np.linspace(0, 1, m)
        for counter in trange(10, desc=f"[n: {n}] [m: {m}]"):
            t = np.sort(rng.choice(np.arange(1, 20) / 20,size = n,replace = False))
            p = rng.integers(1, 9, size = n).astype(float)
            
            for alg in ["milp"]:
                start_time = time()
                res = find_optimal_partition(t, p, tt, c, X, method=alg, check=False, mip_rel_gap=0.1)
                end_time = time()

                res_stats["n"].append(n)
                res_stats["m"].append(m)
                res_stats["counter"].append(counter)
                res_stats["thresholds"].append(t)
                res_stats["priors"].append(res["priors_normalized"])
                res_stats["c"].append(c)
                res_stats["tt"].append(tt)
                res_stats["alg"].append(alg)
                res_stats["time"].append(end_time - start_time)
                res_stats["partition"].append(res["partition"])
                res_stats["loss"].append(res["loss"])
                res_stats["n_vars"].append(res["n_vars"])
                res_stats["n_constrs"].append(res["n_constrs"])

[n: 8] [m: 10]:   0%|          | 0/10 [00:00<?, ?it/s]

Running HiGHS 1.12.0 (git hash: 4f96ee8): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 1244 rows; 600 cols; 8808 nonzeros; 440 integer variables (412 binary)
Coefficient ranges:
  Matrix  [2e-04, 1e+00]
  Cost    [1e-01, 1e-01]
  Bound   [1e+00, 1e+00]
  RHS     [1e-03, 1e+00]
Presolving model
1001 rows, 530 cols, 5330 nonzeros  0s
874 rows, 470 cols, 4963 nonzeros  0s
Presolve reductions: rows 874(-370); columns 470(-130); nonzeros 4963(-3845) 

Solving MIP model with:
   874 rows
   470 cols (343 binary, 0 integer, 0 implied int., 127 continuous, 0 domain fixed)
   4963 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      | 

[n: 8] [m: 10]:  10%|█         | 1/10 [00:17<02:34, 17.20s/it]

     10284       0      4464 100.00%   0.0990041973    0.1                1.00%     5066    121    505    287555    17.2s

Solving report
  Status            Optimal
  Primal bound      0.0999999999999
  Dual bound        0.0990041973173
  Gap               0.996% (tolerance: 1%)
  P-D integral      11.2148914282
  Solution status   feasible
                    0.0999999999999 (objective)
                    0 (bound viol.)
                    2.22364349156e-11 (int. viol.)
                    0 (row viol.)
  Timing            17.18
  Max sub-MIP depth 4
  Nodes             10284
  Repair LPs        0
  LP iterations     287555
                    39783 (strong br.)
                    44642 (separation)
                    14760 (heuristics)
Running HiGHS 1.12.0 (git hash: 4f96ee8): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 1340 rows; 632 cols; 9592 nonzeros; 472 integer variables (444 binary)
Coefficient ranges:
  Matrix  [8e-05, 1e+00]
  Cost    [1e-01, 1e-01]
  Bound

[n: 8] [m: 10]:  20%|██        | 2/10 [00:40<02:44, 20.52s/it]

...

[n: 8] [m: 10]:  60%|██████    | 6/10 [01:58<01:18, 19.50s/it]

      9346       0      4081 100.00%   0.0990016991    0.1                1.00%     4700     81   1311    255954    15.4s

Solving report
  Status            Optimal
  Primal bound      0.0999999999997
  Dual bound        0.099001699064
  Gap               0.998% (tolerance: 1%)
  P-D integral      7.61790842965
  Solution status   feasible
                    0.0999999999997 (objective)
                    0 (bound viol.)
                    9.90341142426e-12 (int. viol.)
                    0 (row viol.)
  Timing            15.43
  Max sub-MIP depth 3
  Nodes             9346
  Repair LPs        0
  LP iterations     255954
                    48871 (strong br.)
                    39422 (separation)
                    12970 (heuristics)
Running HiGHS 1.12.0 (git hash: 4f96ee8): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 1196 rows; 584 cols; 8344 nonzeros; 424 integer variables (396 binary)
Coefficient ranges:
  Matrix  [3e-05, 1e+00]
  Cost    [1e-01, 1e-01]
  Bound  

[n: 8] [m: 10]:  60%|██████    | 6/10 [02:18<01:32, 23.09s/it]

     12992       0      5771 100.00%   0.0990041945    0.1                1.00%     4839    158   1402    326282    20.3s

Solving report
  Status            Optimal
  Primal bound      0.1
  Dual bound        0.0990041945393
  Gap               0.996% (tolerance: 1%)
  P-D integral      12.3022168403
  Solution status   feasible
                    0.1 (objective)
                    0 (bound viol.)
                    1.09912079438e-14 (int. viol.)
                    0 (row viol.)
  Timing            20.28
  Max sub-MIP depth 3
  Nodes             12992
  Repair LPs        0
  LP iterations     326282
                    49026 (strong br.)
                    49836 (separation)
                    16966 (heuristics)


KeyboardInterrupt: 

In [77]:
# pd.DataFrame(res_stats)

In [16]:
import pandas as pd

df = pd.DataFrame(res_stats)
# df.to_csv("milp.csv")
# df = pd.read_csv("milp.csv")
df[df["alg"]=="milp"]

,n,m,counter,thresholds,priors,c,tt,alg,time,partition,loss,n_vars,n_constrs
0,8,10,0,"[0.05, 0.1, 0.4, 0.5, 0.55, 0.6, 0.65, 0.9]","[0.047619047619047616, 0.16666666666666666, 0....",0.1,0.1,milp,17.199446,"[[0, 4], [1], [2, 5], [3, 6], [7]]",0.1,600,1244
1,8,10,1,"[0.15, 0.25, 0.3, 0.35, 0.5, 0.75, 0.85, 0.95]","[0.175, 0.125, 0.2, 0.2, 0.05, 0.05, 0.075, 0....",0.1,0.1,milp,22.845998,"[[0, 2, 4, 5], [1], [3], [6, 7]]",0.1,632,1340
2,8,10,2,"[0.25, 0.3, 0.35, 0.5, 0.55, 0.65, 0.7, 0.75]","[0.11764705882352941, 0.17647058823529413, 0.0...",0.1,0.1,milp,17.602579,"[[0], [1, 5], [2], [3, 7], [4, 6]]",0.1,624,1316
3,8,10,3,"[0.1, 0.15, 0.2, 0.3, 0.55, 0.65, 0.9, 0.95]","[0.1, 0.075, 0.15, 0.1, 0.075, 0.2, 0.15, 0.15]",0.1,0.1,milp,19.506531,"[[0, 5, 7], [1, 2, 6], [3, 4]]",0.1,600,1244
4,8,10,4,"[0.05, 0.15, 0.25, 0.45, 0.65, 0.8, 0.9, 0.95]","[0.03571428571428571, 0.07142857142857142, 0.2...",0.1,0.1,milp,25.634760,"[[0, 2, 4], [1, 3, 5, 6, 7]]",0.1,648,1388
5,8,10,5,"[0.05, 0.1, 0.15, 0.2, 0.25, 0.45, 0.6, 0.95]","[0.1951219512195122, 0.04878048780487805, 0.19...",0.1,0.1,milp,15.443060,"[[0, 1, 2, 3, 4, 5, 6, 7]]",0.1,536,1052


In [17]:
df_agg = df[df["m"] < 200].groupby(["alg", "m"], as_index=False).mean(1)
df_agg

,alg,m,n,counter,c,tt,time,loss,n_vars,n_constrs
0,milp,10,8.0,2.5,0.1,0.1,19.705396,0.1,606.666667,1264.0


In [19]:
import plotly.express as px

In [20]:
px.line(df_agg[df_agg], x="m", y="time", color='alg', labels={"time": "avg time (s)"}, markers=True)

ValueError: Boolean array expected for the condition, not object

In [24]:
# X = np.arange(0., 1. + 1e-4, 1e-4).round(4)
X = np.linspace(0,1,11)

thresholds = np.linspace(0.0, 1.0, 5)
priors = np.array([0.097, 0.015, 0.015, 0.6286, 0.2444])
p = (0.2444/0.25 - 0.97) / (0.4888/0.25 - 0.97)
diff = 1-2*p
priors = np.array([0.097*diff/0.97, p, p, 0.6286*diff/0.97, 0.2444*diff/0.97])
assert(abs(np.sum(priors) - 1) < 1e-6)
c = 1.
tt = 0.1

milp = find_optimal_partition(thresholds, priors, tt, c, X, method="milp")

TypeError: 'dict' object is not callable

In [22]:
milp["partition"]

[[0, 1, 3, 4], [2]]